# Comparaison des modèles et architectures

Ce notebook lit les résultats de validation. Les cibles s'arrêtent à septembre 2024. Il n'entraîne pas de modèles et ne charge pas le test final. Les sorties enregistrées proviennent de l'extraction de vérification; réexécuter avec ton environnement pour actualiser les résultats.

In [1]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = Path(os.environ['CACIB_PROJECT_ROOT']) if os.environ.get('CACIB_PROJECT_ROOT') else next(p for p in candidates if (p / 'finish_project.py').exists())
sys.path.insert(0, str(ROOT))
from cacib_lab.data import lab_root, read_json, load_snapshot
from cacib_lab.evaluation import load_run, metrics, MODEL_LABELS
print('Projet :', ROOT)


Projet : /Users/hadisakho/PycharmProjects/cacib-hicp


In [2]:
run, metadata, predictions = load_run()
assert metadata['phase'] == 'validation'
assert predictions.target.max() <= '2024-09'
print('Comparaison :', run.name)
print('Cibles :', metadata['targets'])
scores = metrics(predictions)
display(scores.query('horizon == 3'))


Comparaison : validation_20261027T132523_420807Z
Cibles : ['2014-01', '2024-09']


,model,horizon,expected,succeeded,failures,eligible,mae_pp,rmse_pp,bias_pp,origins_with_fallback
23,ensemble__ets_branches,3,129,129,0,True,0.530633,0.720691,-0.138802,0
8,direct__ets_additive,3,129,129,0,True,0.563643,0.740114,-0.105284,0
2,branches__ets_additive,3,129,129,0,True,0.521353,0.745073,-0.172319,0
17,direct__ridge_ppi,3,129,129,0,True,0.607079,0.763800,-0.060780,0
14,direct__ridge,3,129,129,0,True,0.614259,0.789595,0.032207,0
5,countries__seasonal_mean,3,129,129,0,True,0.577891,0.843930,-0.170334,0
20,direct__seasonal_mean,3,129,129,0,True,0.579742,0.849560,-0.180618,0
11,direct__flat_yoy,3,129,129,0,True,0.675036,0.935864,-0.031577,0


## Erreur à trois mois

Le critère principal est le RMSE en points de pourcentage. MAE, biais et échecs complètent ce classement.

In [3]:
h3 = scores.query('horizon == 3 and eligible').sort_values('rmse_pp', ascending=False)
fig = go.Figure(go.Bar(x=h3.rmse_pp, y=h3.model.map(MODEL_LABELS), orientation='h', marker_color='#087f75'))
fig.update_layout(template='plotly_white', title='RMSE de validation à trois mois', xaxis_title='Points de pourcentage', margin=dict(l=260), height=460)
fig.show(renderer='plotly_mimetype')


In [4]:
display(pd.read_csv(run / 'metrics_by_period.csv').query('horizon == 3'))
display(pd.read_csv(run / 'metrics_by_crosses_january.csv').query('horizon == 3'))


,model,horizon,expected,succeeded,failures,eligible,mae_pp,rmse_pp,bias_pp,origins_with_fallback,period
16,branches__ets_additive,3,72,72,0,True,0.344576,0.444340,-0.041774,0,2014-2019
17,direct__seasonal_mean,3,72,72,0,True,0.360260,0.454103,0.065007,0,2014-2019
18,direct__flat_yoy,3,72,72,0,True,0.343912,0.454840,-0.008068,0,2014-2019
19,countries__seasonal_mean,3,72,72,0,True,0.358023,0.462871,0.066845,0,2014-2019
20,ensemble__ets_branches,3,72,72,0,True,0.370694,0.470924,-0.038148,0,2014-2019
21,direct__ets_additive,3,72,72,0,True,0.415367,0.533732,-0.034523,0,2014-2019
22,direct__ridge,3,72,72,0,True,0.444483,0.540100,0.205501,0,2014-2019
23,direct__ridge_ppi,3,72,72,0,True,0.435969,0.541824,0.157464,0,2014-2019
40,direct__seasonal_mean,3,12,12,0,True,0.449372,0.550890,0.333809,0,2020
41,branches__ets_additive,3,12,12,0,True,0.451598,0.556032,0.122826,0,2020


,model,horizon,expected,succeeded,failures,eligible,mae_pp,rmse_pp,bias_pp,origins_with_fallback,crosses_january
16,ensemble__ets_branches,3,96,96,0,True,0.479188,0.628011,-0.201773,0,False
17,direct__ets_additive,3,96,96,0,True,0.499939,0.647609,-0.177824,0,False
18,branches__ets_additive,3,96,96,0,True,0.479061,0.662793,-0.225722,0,False
19,direct__ridge_ppi,3,96,96,0,True,0.546666,0.689735,-0.113354,0,False
20,direct__ridge,3,96,96,0,True,0.539191,0.693356,-0.034990,0,False
21,countries__seasonal_mean,3,96,96,0,True,0.537376,0.792044,-0.186487,0,False
22,direct__seasonal_mean,3,96,96,0,True,0.538341,0.795081,-0.178964,0,False
23,direct__flat_yoy,3,96,96,0,True,0.651515,0.913338,-0.030293,0,False
40,ensemble__ets_branches,3,33,33,0,True,0.680291,0.939695,0.044386,0,True
41,branches__ets_additive,3,33,33,0,True,0.644384,0.944522,-0.016968,0,True


## Prévisions hors échantillon

La courbe compare les réalisations aux prévisions faites trois mois auparavant. Elle ne présente pas des valeurs ajustées sur les mêmes observations.

In [5]:
best = scores.query('horizon == 3 and eligible').sort_values('rmse_pp').iloc[0]['model']
x = predictions.query('model == @best and horizon == 3').sort_values('target')
fig = go.Figure()
fig.add_scatter(x=x.target, y=x.actual_yoy, name='Inflation observée', line=dict(color='#142e40'))
fig.add_scatter(x=x.target, y=x.predicted_yoy, name=MODEL_LABELS[best], line=dict(color='#087f75'))
fig.update_layout(template='plotly_white', title='Prévisions à h=3 et réalisations', yaxis_title='Inflation annuelle (%)', height=420)
fig.show(renderer='plotly_mimetype')


In [6]:
display(pd.read_csv(run / 'worst_errors_h3.csv').query('model == @best'))
bootstrap_path = run / 'paired_loss_bootstrap.csv'
if bootstrap_path.exists():
    display(pd.read_csv(bootstrap_path))
else:
    print('Exécuter finish_project.py analyze pour produire le bootstrap.')


,origin,target,horizon,model,status,predicted_index,actual_index,predicted_yoy,actual_yoy,error_pp,fallback_count,crosses_january,failure,absolute_error
5,2021-12,2022-03,3,ensemble__ets_branches,ok,86.590455,88.90,4.653680,7.445008,-2.791328,0,True,NaN,2.791328
21,2022-10,2023-01,3,ensemble__ets_branches,ok,95.394340,93.41,10.949453,8.641545,2.307909,0,True,NaN,2.307909
33,2021-08,2021-11,3,ensemble__ets_branches,ok,83.972260,85.36,3.160024,4.864865,-1.704841,0,False,NaN,1.704841
36,2022-07,2022-10,3,ensemble__ets_branches,ok,92.608447,94.00,8.976756,10.614262,-1.637506,0,False,NaN,1.637506
37,2021-09,2021-12,3,ensemble__ets_branches,ok,84.414215,85.72,3.360126,4.958981,-1.598855,0,False,NaN,1.598855


,model,benchmark,targets,mean_squared_loss_difference,ci95_low,ci95_high,block_months,interpretation
0,branches__ets_additive,direct__ets_additive,129,0.007365,-0.131953,0.190252,6,negative_favors_candidate
1,countries__seasonal_mean,direct__ets_additive,129,0.164448,-0.177955,0.631954,6,negative_favors_candidate
2,direct__ets_additive,direct__ets_additive,129,0.000000,0.000000,0.000000,6,negative_favors_candidate
3,direct__flat_yoy,direct__ets_additive,129,0.328072,0.035624,0.730411,6,negative_favors_candidate
4,direct__ridge,direct__ets_additive,129,0.075690,-0.067269,0.234778,6,negative_favors_candidate
5,direct__ridge_ppi,direct__ets_additive,129,0.035622,-0.130596,0.194503,6,negative_favors_candidate
6,direct__seasonal_mean,direct__ets_additive,129,0.173984,-0.171790,0.640509,6,negative_favors_candidate
7,ensemble__ets_branches,direct__ets_additive,129,-0.028374,-0.094651,0.054282,6,negative_favors_candidate


## Incertitude et couverture

Les intervalles à chaque origine utilisent seulement des erreurs dont les cibles sont déjà observées. La couverture empirique peut différer de 80 % ou 95 %, particulièrement pendant les ruptures.

In [7]:
interval_path = run / 'interval_metrics.csv'
if interval_path.exists():
    display(pd.read_csv(interval_path).query('horizon == 3'))
else:
    print('Exécuter finish_project.py analyze pour produire les intervalles.')


,model,horizon,level,evaluated,coverage,mean_width,mean_interval_score
4,branches__ets_additive,3,0.80,89,0.786517,1.940335,2.796825
5,branches__ets_additive,3,0.95,89,0.921348,3.141961,3.787583
10,countries__seasonal_mean,3,0.80,89,0.764045,2.159936,3.489304
11,countries__seasonal_mean,3,0.95,89,0.910112,3.552154,4.886905
16,direct__ets_additive,3,0.80,89,0.808989,2.173221,2.856192
17,direct__ets_additive,3,0.95,89,0.921348,2.857170,3.503629
22,direct__flat_yoy,3,0.80,89,0.685393,2.377889,3.802397
23,direct__flat_yoy,3,0.95,89,0.910112,4.026133,4.719322
28,direct__ridge,3,0.80,89,0.752809,2.036769,3.168978
29,direct__ridge,3,0.95,89,0.898876,3.082795,4.344869


## Sensibilité : dix ans d'apprentissage

La fenêtre croissante et la fenêtre de 120 mois sont comparées uniquement sur la validation.

In [8]:
pointer = lab_root() / 'latest_validation_120.json'
if pointer.exists():
    run120, meta120, pred120 = load_run(window=120)
    assert meta120['targets'] == metadata['targets']
    comparison = scores.query('horizon == 3')[['model', 'rmse_pp', 'mae_pp']].merge(
        metrics(pred120).query('horizon == 3')[['model', 'rmse_pp', 'mae_pp']],
        on='model', suffixes=('_expanding', '_120months'))
    display(comparison.sort_values('rmse_pp_120months'))
else:
    print('Exécuter compare --window 120 puis analyze --window 120.')


,model,rmse_pp_expanding,mae_pp_expanding,rmse_pp_120months,mae_pp_120months
0,ensemble__ets_branches,0.720691,0.530633,0.707519,0.503139
1,direct__ets_additive,0.740114,0.563643,0.711389,0.522664
2,branches__ets_additive,0.745073,0.521353,0.733811,0.512112
3,direct__ridge_ppi,0.763800,0.607079,0.780300,0.622813
5,countries__seasonal_mean,0.843930,0.577891,0.843930,0.577891
6,direct__seasonal_mean,0.849560,0.579742,0.849560,0.579742
4,direct__ridge,0.789595,0.614259,0.866085,0.645566
7,direct__flat_yoy,0.935864,0.675036,0.935864,0.675036


## Décision avant test

Examiner RMSE, MAE, biais, stabilité et couverture. Le meilleur score moyen ne démontre pas à lui seul une supériorité statistique. Figer ensuite la procédure avec `freeze` ou `freeze --window 120`, puis utiliser la commande distincte `test`. Ce notebook ne sélectionne rien à partir du test final.